# UC-06 Etape 3 - Exploration des donnees avec PySpark
Team **MaxMag** : Maxime DANINO - Awadi BEDJA - Gurvan Godin

Sources explorees (retenues a l'etape 2) : **store_sales**, **item**, **store**, **date_dim**, lues dans la zone Bronze de HDFS.
Pour chaque source : chargement, schema, echantillon, comptage, valeurs manquantes, doublons, valeurs atypiques.

Methode : chaque fichier est lu **tout en texte** avec les noms de colonnes du schema TPCx-BB (`schemas.py`), puis
converti vers les types cibles. Les valeurs non convertibles sont comptees : c'est le controle des types.

In [1]:
import os, sys
sys.path.insert(0, os.path.expanduser('~/retail-intelligence/tp3/scripts'))
from pyspark.sql import functions as F
import config as C, schemas as SC
spark = C.get_spark('UC06-etape3-exploration')
print('Spark', spark.version, '|', spark.sparkContext.master)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/19 11:23:00 WARN Utils: Your hostname, PICOXIA-03, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/19 11:23:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/19 11:23:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.1 | local[*]


In [2]:
def lire(table, lot='initial'):
    """Lecture brute (texte) d'une table Bronze, avec les noms de colonnes du schema."""
    return spark.read.option('sep', '|').schema(SC.schema_texte(table)).csv(C.bronze(table, lot))

def typer(df, table):
    """Conversion vers les types cibles (sans rien supprimer)."""
    return df.select([F.col(n).cast(t).alias(n) for n, t in SC.SOURCES[table]['colonnes']])

def controle_types(brut, table):
    """Nombre de valeurs non vides que la conversion transforme en null (= type incorrect)."""
    exprs = [F.sum((F.col(n).isNotNull() & F.col(n).cast(t).isNull()).cast('int')).alias(n)
             for n, t in SC.SOURCES[table]['colonnes']]
    r = brut.agg(*exprs).collect()[0].asDict()
    return {k: v for k, v in r.items() if v}

def valeurs_manquantes(df):
    r = df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns]).collect()[0].asDict()
    return {k: v for k, v in r.items() if v}

def profil(table, lot='initial'):
    brut = lire(table, lot)
    n = brut.count()
    print(f'{table} / {lot} : {n} lignes, {len(brut.columns)} colonnes')
    print('valeurs manquantes (null) :', valeurs_manquantes(brut) or 'aucune')
    print('valeurs non convertibles vers le type cible :', controle_types(brut, table) or 'aucune')
    print('lignes entierement identiques en double :', n - brut.distinct().count())
    return typer(brut, table)

## 1. store_sales (ventes en magasin)

In [3]:
ventes = profil('store_sales')
ventes.printSchema()
ventes.select('ss_sold_date_sk','ss_store_sk','ss_item_sk','ss_ticket_number','ss_quantity','ss_sales_price','ss_ext_sales_price','ss_net_paid','ss_net_profit').show(5)

store_sales / initial : 667579 lignes, 23 colonnes


valeurs manquantes (null) : {'ss_cdemo_sk': 238, 'ss_hdemo_sk': 295, 'ss_addr_sk': 448, 'ss_promo_sk': 158}


valeurs non convertibles vers le type cible : aucune


lignes entierement identiques en double : 0
root
 |-- ss_sold_date_sk: integer (nullable = true)
 |-- ss_sold_time_sk: integer (nullable = true)
 |-- ss_item_sk: integer (nullable = true)
 |-- ss_customer_sk: integer (nullable = true)
 |-- ss_cdemo_sk: integer (nullable = true)
 |-- ss_hdemo_sk: integer (nullable = true)
 |-- ss_addr_sk: integer (nullable = true)
 |-- ss_store_sk: integer (nullable = true)
 |-- ss_promo_sk: integer (nullable = true)
 |-- ss_ticket_number: integer (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_list_price: decimal(7,2) (nullable = true)
 |-- ss_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(7,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_ext_list_price: decimal(7,2) (nullable = true)
 |-- ss_ext_tax: decimal(7,2) (nullable = true)
 |-- ss_coupon_amt: 

+---------------+-----------+----------+----------------+-----------+--------------+------------------+-----------+-------------+
|ss_sold_date_sk|ss_store_sk|ss_item_sk|ss_ticket_number|ss_quantity|ss_sales_price|ss_ext_sales_price|ss_net_paid|ss_net_profit|
+---------------+-----------+----------+----------------+-----------+--------------+------------------+-----------+-------------+
|          36890|         10|     10001|               0|         31|         23.08|            715.48|     715.48|       157.48|
|          36890|         10|     10000|               0|         54|         83.34|           4500.36|    4500.36|      -856.98|
|          36890|         10|     15556|               0|         39|         83.24|           3246.36|    2629.55|      -470.56|
|          36890|          4|      4597|               1|         20|        133.40|           2668.00|    2668.00|      1180.40|
|          36890|          4|      3258|               1|         76|         11.85|      

In [4]:
# Doublons de la cle candidate ticket + produit : sont-ce de vraies copies ?
cle = ['ss_ticket_number', 'ss_item_sk']
dup = ventes.groupBy(*cle).count().filter('count > 1')
print('couples ticket+produit repetes :', dup.count(), '| lignes concernees :', dup.agg(F.sum('count')).collect()[0][0])
exemple = dup.orderBy('ss_ticket_number').first()
ventes.filter((F.col('ss_ticket_number') == exemple.ss_ticket_number) & (F.col('ss_item_sk') == exemple.ss_item_sk)) \
      .select(*cle, 'ss_sold_time_sk', 'ss_promo_sk', 'ss_quantity', 'ss_sales_price', 'ss_net_paid').show()

couples ticket+produit repetes : 4038 | lignes concernees : 8174


+----------------+----------+---------------+-----------+-----------+--------------+-----------+
|ss_ticket_number|ss_item_sk|ss_sold_time_sk|ss_promo_sk|ss_quantity|ss_sales_price|ss_net_paid|
+----------------+----------+---------------+-----------+-----------+--------------+-----------+
|              44|     10003|          27946|        283|         31|         53.16|    1647.96|
|              44|     10003|          27946|         22|         46|          1.60|       5.89|
+----------------+----------+---------------+-----------+-----------+--------------+-----------+



In [5]:
# Valeurs atypiques : statistiques des mesures
ventes.select('ss_quantity','ss_sales_price','ss_ext_sales_price','ss_net_paid','ss_net_profit').summary('min','25%','50%','75%','max','mean').show()
ventes.agg(F.sum((F.col('ss_quantity') <= 0).cast('int')).alias('quantite_<=0'),
           F.sum((F.col('ss_sales_price') <= 0).cast('int')).alias('prix_vente_<=0'),
           F.sum((F.col('ss_net_paid') <= 0).cast('int')).alias('net_paye_<=0'),
           F.sum((F.col('ss_net_profit') < 0).cast('int')).alias('marge_<0'),
           F.sum((F.col('ss_sales_price') > F.col('ss_list_price')).cast('int')).alias('prix_vente_>_prix_catalogue')).show()

+-------+------------------+--------------+------------------+-----------+-------------+
|summary|       ss_quantity|ss_sales_price|ss_ext_sales_price|ss_net_paid|ss_net_profit|
+-------+------------------+--------------+------------------+-----------+-------------+
|    min|                 1|          0.00|              0.00|       0.00|     -9866.60|
|    25%|                26|         10.24|             313.5|      245.7|     -1462.23|
|    50%|                51|         27.69|           1048.32|     878.23|       -377.3|
|    75%|                76|         56.76|            2673.6|    2365.65|         5.82|
|    max|               100|        199.30|          18745.00|   18745.00|      9134.00|
|   mean|50.564014146640325|     37.863697|       1913.217768|1723.435858|  -827.697358|
+-------+------------------+--------------+------------------+-----------+-------------+



+------------+--------------+------------+--------+---------------------------+
|quantite_<=0|prix_vente_<=0|net_paye_<=0|marge_<0|prix_vente_>_prix_catalogue|
+------------+--------------+------------+--------+---------------------------+
|           0|          3324|        3954|  495971|                          0|
+------------+--------------+------------+--------+---------------------------+



In [6]:
# Coherence des montants : regles de calcul TPC-DS verifiees ligne a ligne (tolerance 1 centime)
ecart = lambda a, b: (F.abs(a - b) > 0.01).cast('int')
ventes.agg(
  F.sum(ecart(F.col('ss_ext_sales_price'), F.col('ss_sales_price') * F.col('ss_quantity'))).alias('ext_sales != prix x qte'),
  F.sum(ecart(F.col('ss_ext_list_price'), F.col('ss_list_price') * F.col('ss_quantity'))).alias('ext_list != catalogue x qte'),
  F.sum(ecart(F.col('ss_net_paid'), F.col('ss_ext_sales_price') - F.col('ss_coupon_amt'))).alias('net_paid != ext_sales - coupon'),
  F.sum(ecart(F.col('ss_net_paid_inc_tax'), F.col('ss_net_paid') + F.col('ss_ext_tax'))).alias('net_paid_inc_tax != net_paid + taxe'),
  F.sum(ecart(F.col('ss_net_profit'), F.col('ss_net_paid') - F.col('ss_ext_wholesale_cost'))).alias('profit != net_paid - cout')).show(vertical=True)

-RECORD 0----------------------------------
 ext_sales != prix x qte             | 0   
 ext_list != catalogue x qte         | 0   
 net_paid != ext_sales - coupon      | 0   
 net_paid_inc_tax != net_paid + taxe | 0   
 profit != net_paid - cout           | 0   



**Constats store_sales** (lot initial) :
- 667 579 lignes, 23 colonnes ; toutes les valeurs se convertissent vers les types cibles (entiers, decimal(7,2)).
- Valeurs manquantes seulement sur 4 cles secondaires : client demographique 238, menage 295, adresse 448, promotion 158 (moins de 0,07 %). Les cles date, magasin, produit et ticket ne sont jamais vides.
- Aucune ligne copiee a l'identique. Mais 4 038 couples ticket + produit apparaissent plusieurs fois (8 174 lignes) : meme ticket, meme produit, meme heure, mais promotion, quantite et prix differents. Ce sont des lignes de vente distinctes, a conserver.
- Quantites entre 1 et 100, aucune nulle ou negative ; prix de vente jamais superieur au prix catalogue.
- Atypiques : 3 324 lignes a prix de vente 0 et 3 954 a net paye 0 ou moins (coupon couvrant le montant) ; 74 % des lignes (495 971) ont une marge negative, marge moyenne -827,70. Plausibles dans un jeu synthetique, a conserver et a signaler.
- Les 5 regles de calcul des montants (montant = prix x quantite, net paye = montant - coupon, etc.) sont verifiees sur 100 % des lignes.

## 2. item (produits)

In [7]:
produits = profil('item')
produits.select('i_item_sk','i_item_id','i_category','i_class','i_brand','i_current_price','i_product_name').show(5, truncate=False)
produits.agg(F.countDistinct('i_item_sk').alias('i_item_sk distincts'), F.countDistinct('i_item_id').alias('i_item_id distincts'),
             F.min('i_current_price').alias('prix_min'), F.max('i_current_price').alias('prix_max')).show()
produits.groupBy('i_category').count().orderBy(F.desc('count')).show(truncate=False)

item / initial : 17820 lignes, 22 colonnes


valeurs manquantes (null) : {'i_rec_end_date': 17820, 'i_item_desc': 4, 'i_brand_id': 8, 'i_brand': 6}


valeurs non convertibles vers le type cible : aucune


lignes entierement identiques en double : 0
+---------+----------------+-----------------+---------------------+---------------------------------+---------------+------------------------------------------------+
|i_item_sk|i_item_id       |i_category       |i_class              |i_brand                          |i_current_price|i_product_name                                  |
+---------+----------------+-----------------+---------------------+---------------------------------+---------------+------------------------------------------------+
|0        |AAAAAAAAAAAAAAAA|Sports & Outdoors|Fan Shop             |71TS7NSbvH1YbdiQMG6ttBHKAljiNoIRB|72.29          |8m9n5Q7T33DNWidoA6nWlg6ydmpA1SKOoOJLXiLVb       |
|1        |AAAAAAAAAAAAAAAB|Books            |New                  |qNBimHGBu763udH                  |95.67          |CPB648WB37uUv5OCqjzYe7rH3T                      |
|2        |AAAAAAAAAAAAAAAC|Books            |Education & Reference|ma5                              |62.74         

+-------------------+-------------------+--------+--------+
|i_item_sk distincts|i_item_id distincts|prix_min|prix_max|
+-------------------+-------------------+--------+--------+
|              17820|              17820|    0.10|   99.98|
+-------------------+-------------------+--------+--------+



+------------------------+-----+
|i_category              |count|
+------------------------+-----+
|Music                   |8143 |
|Books                   |5335 |
|Movies & TV             |1286 |
|Electronics             |692  |
|Clothing & Accessories  |610  |
|Home & Kitchen          |476  |
|Sports & Outdoors       |465  |
|Tools & Home Improvement|408  |
|Toys & Games            |405  |
+------------------------+-----+



In [8]:
# Produits sans marque ou sans description
produits.filter(F.col('i_brand').isNull() | F.col('i_item_desc').isNull()).select('i_item_sk','i_category','i_brand','i_brand_id','i_item_desc').show(truncate=40)

+---------+-----------------+-------------------------------+----------+----------------------------------------+
|i_item_sk|       i_category|                        i_brand|i_brand_id|                             i_item_desc|
+---------+-----------------+-------------------------------+----------+----------------------------------------+
|      903|            Music|              pFjxZzS90b7rpeV32|   7970429|                                    NULL|
|     3080|            Music|                           NULL|   3754013|stealthy even escapades behind the th...|
|     3453|            Music|                      AxTkcfnJq|   3792125|                                    NULL|
|     4555|            Books|                           NULL|   5059771|realms since the carefully busy foxes...|
|     8382|            Books|    5kvnaDEXYm3jYYox2qfrhgpE2V2|   3570212|                                    NULL|
|     9607|      Movies & TV|                           NULL|   6330153|foxes may haggle

**Constats item** :
- 17 820 produits, cle i_item_sk unique, i_item_id aussi unique : une seule version par produit (i_rec_end_date toujours vide).
- 9 categories exploitables (Music 8 143, Books 5 335...). Les marques, descriptions et noms de produits sont des textes aleatoires : utilisables comme libelles, pas pour des analyses par marque.
- 6 marques et 8 identifiants de marque vides, 4 descriptions vides ; la marque peut etre vide alors que son identifiant existe.
- Prix courant entre 0,10 et 99,98.

## 3. store (magasins)

In [9]:
magasins = profil('store')
magasins.select('s_store_sk','s_store_id','s_store_name','s_city','s_state','s_number_employees','s_floor_space','s_closed_date_sk').show(truncate=False)
magasins.agg(F.countDistinct('s_store_id').alias('s_store_id distincts')).show()

store / initial : 12 lignes, 29 colonnes


valeurs manquantes (null) : {'s_rec_end_date': 12}


valeurs non convertibles vers le type cible : aucune


lignes entierement identiques en double : 0


+----------+----------------+------------+------------+-------+------------------+-------------+----------------+
|s_store_sk|s_store_id      |s_store_name|s_city      |s_state|s_number_employees|s_floor_space|s_closed_date_sk|
+----------+----------------+------------+------------+-------+------------------+-------------+----------------+
|0         |AAAAAAAAAAAAAAAA|Forest      |Post Oak    |CO     |217               |8891512      |71238           |
|1         |AAAAAAAAAAAAAAAB|Hillside    |Jerusalem   |TN     |244               |8751339      |18994           |
|2         |AAAAAAAAAAAAAAAC|Sherwood    |Mount Vernon|CA     |266               |5913334      |8900            |
|3         |AAAAAAAAAAAAAAAD|Pioneer     |Austin      |TX     |221               |5777026      |69902           |
|4         |AAAAAAAAAAAAAAAE|Elmwood     |Lakeside    |KY     |254               |7508523      |79617           |
|5         |AAAAAAAAAAAAAAAF|Egypt       |Redland     |KY     |215               |694244

**Constats store** :
- 12 magasins, cles s_store_sk et s_store_id uniques, aucune valeur non convertible.
- s_rec_end_date toujours vide.
- s_closed_date_sk est renseigne pour les 12 magasins, mais correspond a des dates sans sens (de 1917 a 2141), alors que tous les magasins vendent de 2001 a 2005 : colonne inutilisable, a ne pas interpreter comme une fermeture.

## 4. date_dim (calendrier)

In [10]:
dates = profil('date_dim')
dates.select('d_date_sk','d_date','d_year','d_moy','d_qoy','d_day_name','d_weekend','d_holiday').show(5)
dates.agg(F.min('d_date'), F.max('d_date'), F.min('d_date_sk'), F.max('d_date_sk')).show()
# d_date_sk = nombre de jours depuis le 01/01/1900 ? (dans TPC-DS d'origine c'est un jour julien, 2415021 pour cette date)
dates.agg((F.max('d_date_sk') - F.min('d_date_sk') + 1).alias('plage_sk'), F.count('*').alias('jours'),
          F.sum((F.datediff('d_date', F.lit('1900-01-01')) != F.col('d_date_sk')).cast('int')).alias('sk_differents_jours_depuis_1900')).show()
dates.groupBy('d_holiday').count().show()

date_dim / initial : 109573 lignes, 28 colonnes


valeurs manquantes (null) : aucune


valeurs non convertibles vers le type cible : aucune


lignes entierement identiques en double : 0


+---------+----------+------+-----+-----+----------+---------+---------+
|d_date_sk|    d_date|d_year|d_moy|d_qoy|d_day_name|d_weekend|d_holiday|
+---------+----------+------+-----+-----+----------+---------+---------+
|        0|1900-01-01|  1900|    1|    1|    Monday|        N|        Y|
|        1|1900-01-02|  1900|    1|    1|   Tuesday|        N|        Y|
|        2|1900-01-03|  1900|    1|    1| Wednesday|        N|        Y|
|        3|1900-01-04|  1900|    1|    1|  Thursday|        N|        Y|
|        4|1900-01-05|  1900|    1|    1|    Friday|        N|        Y|
+---------+----------+------+-----+-----+----------+---------+---------+
only showing top 5 rows
+-----------+-----------+--------------+--------------+
|min(d_date)|max(d_date)|min(d_date_sk)|max(d_date_sk)|
+-----------+-----------+--------------+--------------+
| 1900-01-01| 2199-12-31|             0|        109572|
+-----------+-----------+--------------+--------------+



+--------+------+-------------------------------+
|plage_sk| jours|sk_differents_jours_depuis_1900|
+--------+------+-------------------------------+
|  109573|109573|                              0|
+--------+------+-------------------------------+

+---------+------+
|d_holiday| count|
+---------+------+
|        Y|109573|
+---------+------+



In [11]:
# Date de fermeture des magasins : s_closed_date_sk renseigne pour les 12 magasins, que represente-t-il ?
magasins.join(dates, magasins.s_closed_date_sk == dates.d_date_sk, 'left') \
        .select('s_store_sk', 's_store_name', 's_rec_start_date', 's_closed_date_sk', F.col('d_date').alias('date_fermeture')).orderBy('s_store_sk').show()

+----------+------------+----------------+----------------+--------------+
|s_store_sk|s_store_name|s_rec_start_date|s_closed_date_sk|date_fermeture|
+----------+------------+----------------+----------------+--------------+
|         0|      Forest|      2000-08-02|           71238|    2095-01-16|
|         1|    Hillside|      2000-07-03|           18994|    1952-01-03|
|         2|    Sherwood|      2000-09-01|            8900|    1924-05-15|
|         3|     Pioneer|      2000-06-02|           69902|    2091-05-21|
|         4|     Elmwood|      2000-07-03|           79617|    2117-12-26|
|         5|       Egypt|      2000-03-03|           88139|    2141-04-26|
|         6|  Bloomfield|      2000-01-03|           15761|    1943-02-26|
|         7|     Buffalo|      2000-03-03|           71446|    2095-08-12|
|         8|Williamstown|      2000-06-02|           50689|    2038-10-13|
|         9|  Germantown|      2000-11-01|           25131|    1968-10-22|
|        10|      Avalon|

**Constats date_dim** :
- 109 573 jours, du 01/01/1900 au 31/12/2199, sans trou ; aucune valeur manquante.
- d_date_sk = nombre de jours depuis le 01/01/1900 (0 pour cette date), verifie sur toutes les lignes. Dans TPC-DS d'origine c'est un jour julien : le kit utilise donc une numerotation propre.
- d_holiday vaut Y pour tous les jours : colonne inutilisable pour distinguer les jours feries.

## 5. Liens entre les sources (preparation des jointures)

In [12]:
def orphelins(fait, cle_fait, dim, cle_dim):
    return fait.join(dim.select(F.col(cle_dim).alias(cle_fait)), cle_fait, 'left_anti').count()
print('ventes dont le produit est absent de item (lot initial) :', orphelins(ventes, 'ss_item_sk', produits, 'i_item_sk'))
print('ventes dont le magasin est absent de store             :', orphelins(ventes, 'ss_store_sk', magasins, 's_store_sk'))
print('ventes dont la date est absente de date_dim            :', orphelins(ventes, 'ss_sold_date_sk', dates, 'd_date_sk'))
periode = ventes.join(dates, ventes.ss_sold_date_sk == dates.d_date_sk).agg(F.min('d_date').alias('premiere_vente'), F.max('d_date').alias('derniere_vente'))
periode.show()

ventes dont le produit est absent de item (lot initial) : 0


ventes dont le magasin est absent de store             : 0


ventes dont la date est absente de date_dim            : 0


+--------------+--------------+
|premiere_vente|derniere_vente|
+--------------+--------------+
|    2001-01-01|    2005-12-13|
+--------------+--------------+



In [13]:
# Lots refresh : nouvelles lignes ou recouvrement ?
ventes_r = typer(lire('store_sales', 'refresh'), 'store_sales')
produits_r = typer(lire('item', 'refresh'), 'item')
print('tickets du refresh deja presents dans le lot initial :', ventes_r.select('ss_ticket_number').distinct().join(ventes.select('ss_ticket_number').distinct(), 'ss_ticket_number').count())
print('produits du refresh deja presents dans le lot initial :', produits_r.join(produits.select('i_item_sk'), 'i_item_sk').count(), 'sur', produits_r.count())
tous_produits = produits.unionByName(produits_r)
print('ventes du refresh dont le produit est absent de item (initial + refresh) :', orphelins(ventes_r, 'ss_item_sk', tous_produits, 'i_item_sk'))
ventes_r.join(dates, ventes_r.ss_sold_date_sk == dates.d_date_sk).agg(F.min('d_date'), F.max('d_date'), F.count('*')).show()

tickets du refresh deja presents dans le lot initial : 0
produits du refresh deja presents dans le lot initial : 0 sur 180


ventes du refresh dont le produit est absent de item (initial + refresh) : 0


+-----------+-----------+--------+
|min(d_date)|max(d_date)|count(1)|
+-----------+-----------+--------+
| 2005-12-13| 2005-12-31|    6632|
+-----------+-----------+--------+



**Constats sur les liens** :
- Aucune vente orpheline : tous les produits, magasins et dates des ventes existent dans item, store et date_dim. Les jointures ventes-produits, ventes-magasins et ventes-dates sont donc possibles sans perte.
- Les ventes du lot initial vont du 01/01/2001 au 13/12/2005, celles du lot refresh du 13/12/2005 au 31/12/2005.
- Les lots ne se recouvrent pas : aucun ticket ni aucun produit du refresh n'existe dans le lot initial. Les ventes du refresh ne trouvent leur produit qu'en reunissant les deux lots de item : il faudra charger item initial + refresh.

In [14]:
spark.stop()